<a href="https://colab.research.google.com/github/Yascaram/Datathon-Fase-5-/blob/main/An%C3%A1lise_completa_indicadores_Tech5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# %load /content/analise_completa.py
"""
Análise Completa dos Indicadores — Passos Mágicos (2022–2024)
=============================================================
Q1  Adequação do nível (IAN) e defasagem escolar
Q2  Desempenho acadêmico (IDA) por fase e ano
Q3  Engajamento (IEG) × IDA e IPV
Q4  Autoavaliação (IAA) × desempenho real (IDA) e engajamento (IEG)
Q5  Aspectos psicossociais (IPS) e quedas de desempenho
Q6  Aspectos psicopedagógicos (IPP) × defasagem (IAN)
Q7  Ponto de virada (IPV) — fatores mais influentes
Q8  Multidimensionalidade dos indicadores → impacto no INDE
Q10 Efetividade do programa → evolução por Pedra e Fase

Dependências: pandas, numpy, scipy, scikit-learn, openpyxl
  pip install pandas numpy scipy scikit-learn openpyxl
"""

import re
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler


# ══════════════════════════════════════════════════════════════════════════════
# 1. CARGA E LIMPEZA
# ══════════════════════════════════════════════════════════════════════════════

df22 = pd.read_excel("2022.xlsx")
df23 = pd.read_excel("2023.xlsx")
df24 = pd.read_excel("2024.xlsx")


def clean_2022(df):
    d = df.copy()
    d.rename(columns={
        "Idade 22": "Idade", "INDE 22": "INDE", "Pedra 22": "Pedra",
        "Matem": "Mat", "Portug": "Por", "Inglês": "Ing",
        "Defas": "Defasagem", "Fase ideal": "Fase Ideal",
    }, inplace=True)
    d["Ano"] = 2022
    d["Gênero"] = d["Gênero"].map({"Menina": "Feminino", "Menino": "Masculino"})
    d["Fase"] = d["Fase"].map({
        0: "ALFA", 1: "FASE 1", 2: "FASE 2", 3: "FASE 3", 4: "FASE 4",
        5: "FASE 5", 6: "FASE 6", 7: "FASE 7", 8: "FASE 8",
    })
    d["Instituição de ensino"] = d["Instituição de ensino"].replace(
        {"Escola Pública": "Pública", "Rede Decisão": "Privada", "Escola JP II": "Privada"}
    )
    return d


def clean_2023(df):
    d = df.copy()
    d.rename(columns={
        "Nome Anonimizado": "Nome", "INDE 2023": "INDE", "Pedra 2023": "Pedra",
    }, inplace=True)
    d["Ano"] = 2023
    d["Fase"] = d["Fase"].str.upper().str.strip()
    d["Pedra"] = d["Pedra"].replace({"Agata": "Ágata"})
    return d


def clean_2024(df):
    d = df.copy()
    d.rename(columns={
        "Nome Anonimizado": "Nome", "INDE 2024": "INDE", "Pedra 2024": "Pedra",
    }, inplace=True)
    d["Ano"] = 2024
    d["INDE"] = pd.to_numeric(d["INDE"], errors="coerce")

    def normalizar_fase(f):
        f = str(f).strip().upper()
        if f == "ALFA":
            return "ALFA"
        m = re.match(r"^(\d+)", f)
        if m:
            n = int(m.group(1))
            if 1 <= n <= 8:
                return f"FASE {n}"
        return f

    d["Fase"] = d["Fase"].apply(normalizar_fase)
    d["Pedra"] = d["Pedra"].replace({"Agata": "Ágata", "INCLUIR": np.nan})
    return d


df22c = clean_2022(df22)
df23c = clean_2023(df23)
df24c = clean_2024(df24)

COLS = [
    "Ano", "RA", "Nome", "Fase", "Fase Ideal", "Defasagem",
    "Gênero", "Idade", "Ano ingresso", "Instituição de ensino",
    "Pedra", "INDE", "IAA", "IEG", "IPS", "IDA", "IPV", "IAN",
    "Mat", "Por", "Ing", "IPP",
]


def sel(df):
    return df[[c for c in COLS if c in df.columns]].copy()


base = pd.concat([sel(df22c), sel(df23c), sel(df24c)], ignore_index=True)

FASE_ORDER = ["ALFA", "FASE 1", "FASE 2", "FASE 3", "FASE 4",
              "FASE 5", "FASE 6", "FASE 7", "FASE 8"]

PEDRA_ORDER = ["Topázio", "Ametista", "Ágata", "Quartzo"]

SEP = "=" * 65


def titulo(n, texto):
    print(f"\n{SEP}")
    print(f"  Q{n} — {texto}")
    print(SEP)


# ══════════════════════════════════════════════════════════════════════════════
# Q1 — ADEQUAÇÃO DO NÍVEL (IAN) E DEFASAGEM ESCOLAR
# ══════════════════════════════════════════════════════════════════════════════
titulo(1, "ADEQUAÇÃO DO NÍVEL (IAN) E DEFASAGEM ESCOLAR")


def classif_ian(v):
    if v >= 8:  return "Adequado (≥8)"
    if v >= 6:  return "Leve (6–8)"
    if v >= 4:  return "Moderado (4–6)"
    return "Severo (<4)"


base["Nível IAN"] = base["IAN"].apply(classif_ian)

print("\nIAN médio por ano:")
print(base.groupby("Ano")["IAN"].mean().round(3).to_string())

col_order_ian = ["Severo (<4)", "Moderado (4–6)", "Leve (6–8)", "Adequado (≥8)"]
dist_ian = (
    base.groupby(["Ano", "Nível IAN"]).size().unstack(fill_value=0)
    .div(base.groupby("Ano").size(), axis=0).mul(100).round(1)
    .reindex(columns=[c for c in col_order_ian if c in base["Nível IAN"].unique()])
)
print("\nDistribuição de risco IAN por ano (%):")
print(dist_ian.to_string())

print("\nDefasagem média por ano:")
print(base.groupby("Ano")["Defasagem"].mean().round(3).to_string())

sem_def = base.groupby("Ano").apply(
    lambda x: round((x["Defasagem"] == 0).mean() * 100, 1)
)
print("\n% alunos sem defasagem (Defasagem = 0) por ano:")
print(sem_def.to_string())

print("\nIAN médio por fase (todos os anos):")
print(
    base[base["Fase"].isin(FASE_ORDER)]
    .groupby("Fase")["IAN"].mean().reindex(FASE_ORDER).round(3).to_string()
)


# ══════════════════════════════════════════════════════════════════════════════
# Q2 — DESEMPENHO ACADÊMICO (IDA) POR FASE E ANO
# ══════════════════════════════════════════════════════════════════════════════
titulo(2, "DESEMPENHO ACADÊMICO (IDA) — EVOLUÇÃO POR FASE E ANO")

print("\nIDA médio por ano:")
print(base.groupby("Ano")["IDA"].mean().round(3).to_string())

print("\nIDA médio por fase (todos os anos):")
print(
    base[base["Fase"].isin(FASE_ORDER)]
    .groupby("Fase")["IDA"].mean().reindex(FASE_ORDER).round(3).to_string()
)

print("\nIDA médio por fase × ano:")
print(
    base[base["Fase"].isin(FASE_ORDER)]
    .groupby(["Fase", "Ano"])["IDA"].mean().unstack().round(3)
    .reindex(FASE_ORDER).to_string()
)

print("\nNotas médias por disciplina e ano:")
print(base.groupby("Ano")[["Mat", "Por", "Ing"]].mean().round(3).to_string())


# ══════════════════════════════════════════════════════════════════════════════
# Q3 — ENGAJAMENTO (IEG) × IDA e IPV
# ══════════════════════════════════════════════════════════════════════════════
titulo(3, "ENGAJAMENTO (IEG) × DESEMPENHO (IDA) E PONTO DE VIRADA (IPV)")

sub3 = base[["IEG", "IDA", "IPV"]].dropna()
r_ieg_ida, p_ieg_ida = stats.pearsonr(sub3["IEG"], sub3["IDA"])
r_ieg_ipv, p_ieg_ipv = stats.pearsonr(sub3["IEG"], sub3["IPV"])
print(f"\nCorrelação IEG × IDA : r = {r_ieg_ida:.3f}  (p = {p_ieg_ida:.4f})")
print(f"Correlação IEG × IPV : r = {r_ieg_ipv:.3f}  (p = {p_ieg_ipv:.4f})")
print("  → r > 0.5 = correlação positiva forte e estatisticamente significativa")


def classif_ieg(v):
    if v >= 7.5: return "Alto (≥7.5)"
    if v >= 5:   return "Médio (5–7.5)"
    return "Baixo (<5)"


base["Grupo IEG"] = base["IEG"].apply(classif_ieg)
print("\nIDA e IPV médios por grupo de IEG:")
print(
    base.groupby("Grupo IEG")[["IDA", "IPV"]].mean().round(3)
    .reindex(["Alto (≥7.5)", "Médio (5–7.5)", "Baixo (<5)"]).to_string()
)


# ══════════════════════════════════════════════════════════════════════════════
# Q4 — AUTOAVALIAÇÃO (IAA) × IDA e IEG
# ══════════════════════════════════════════════════════════════════════════════
titulo(4, "AUTOAVALIAÇÃO (IAA) × DESEMPENHO REAL (IDA) E ENGAJAMENTO (IEG)")

sub4 = base[["IAA", "IDA", "IEG"]].dropna()
r_iaa_ida, p_iaa_ida = stats.pearsonr(sub4["IAA"], sub4["IDA"])
r_iaa_ieg, p_iaa_ieg = stats.pearsonr(sub4["IAA"], sub4["IEG"])
print(f"\nCorrelação IAA × IDA : r = {r_iaa_ida:.3f}  (p = {p_iaa_ida:.4f})")
print(f"Correlação IAA × IEG : r = {r_iaa_ieg:.3f}  (p = {p_iaa_ieg:.4f})")
print("  → Correlação fraca: autoavaliação pouco alinhada ao desempenho real")

base["Delta_IAA_IDA"] = base["IAA"] - base["IDA"]
print("\nDelta médio (IAA − IDA) por ano — positivo = superestima:")
print(base.groupby("Ano")["Delta_IAA_IDA"].mean().round(3).to_string())


def classif_coerencia(d):
    if abs(d) <= 1: return "Coerente (±1 ponto)"
    if d > 1:       return "Superestima (>1 ponto)"
    return "Subestima (>1 ponto)"


base["Coerência IAA"] = base["Delta_IAA_IDA"].apply(classif_coerencia)
print("\nDistribuição geral de coerência da autoavaliação (%):")
print(base["Coerência IAA"].value_counts(normalize=True).mul(100).round(1).to_string())


# ══════════════════════════════════════════════════════════════════════════════
# Q5 — ASPECTOS PSICOSSOCIAIS (IPS) E QUEDAS DE DESEMPENHO
# ══════════════════════════════════════════════════════════════════════════════
titulo(5, "ASPECTOS PSICOSSOCIAIS (IPS) × IDA, IEG e IPV")

sub5 = base[["IPS", "IDA", "IEG", "IPV"]].dropna()
for col in ["IDA", "IEG", "IPV"]:
    r, p = stats.pearsonr(sub5["IPS"], sub5[col])
    print(f"Correlação IPS × {col} : r = {r:.3f}  (p = {p:.4f})")
print("  → Correlações próximas de zero: IPS tem pouca relação linear direta")


def classif_ips(v):
    if v >= 7.5: return "Alto (≥7.5)"
    if v >= 5:   return "Médio (5–7.5)"
    return "Baixo (<5)"


base["Grupo IPS"] = base["IPS"].apply(classif_ips)
print("\nIDA, IEG e IPV médios por grupo de IPS:")
print(
    base.groupby("Grupo IPS")[["IDA", "IEG", "IPV"]].mean().round(3)
    .reindex(["Alto (≥7.5)", "Médio (5–7.5)", "Baixo (<5)"]).to_string()
)
print("  → IPS baixo está associado a IEG e IPV mais baixos")


# ══════════════════════════════════════════════════════════════════════════════
# Q6 — ASPECTOS PSICOPEDAGÓGICOS (IPP) × DEFASAGEM (IAN)
# ══════════════════════════════════════════════════════════════════════════════
titulo(6, "PSICOPEDAGÓGICO (IPP) × DEFASAGEM (IAN) E DESEMPENHO (IDA)")

sub6 = base[["IPP", "IAN", "IDA"]].dropna()
r_ipp_ian, p_ipp_ian = stats.pearsonr(sub6["IPP"], sub6["IAN"])
r_ipp_ida, p_ipp_ida = stats.pearsonr(sub6["IPP"], sub6["IDA"])
print(f"\nAmostra disponível: {len(sub6)} alunos (2023–2024)")
print(f"IPP médio: {sub6['IPP'].mean():.3f}")
print(f"\nCorrelação IPP × IAN : r = {r_ipp_ian:.3f}  (p = {p_ipp_ian:.4f})")
print(f"Correlação IPP × IDA : r = {r_ipp_ida:.3f}  (p = {p_ipp_ida:.4f})")
print("  → IPP confirma parcialmente a defasagem (IAN) e o desempenho (IDA)")


def classif_ipp(v):
    if v >= 7.5: return "Alto (≥7.5)"
    if v >= 5:   return "Médio (5–7.5)"
    return "Baixo (<5)"


base["Grupo IPP"] = base["IPP"].apply(classif_ipp)
print("\nIAN e IDA médios por grupo de IPP:")
print(
    base.groupby("Grupo IPP")[["IAN", "IDA"]].mean().round(3)
    .reindex(["Alto (≥7.5)", "Médio (5–7.5)", "Baixo (<5)"]).to_string()
)
print("  → IPP alto corresponde a melhor IDA; IAN varia menos entre grupos")


# ══════════════════════════════════════════════════════════════════════════════
# Q7 — PONTO DE VIRADA (IPV): FATORES MAIS INFLUENTES
# ══════════════════════════════════════════════════════════════════════════════
titulo(7, "PONTO DE VIRADA (IPV) — FATORES COMPORTAMENTAIS E ACADÊMICOS")

# INDE é excluído da regressão pois é composto dos demais indicadores
predictors_ipv = ["IDA", "IEG", "IAA", "IPS", "IAN", "IPP"]
sub7 = base[predictors_ipv + ["IPV"]].dropna()

print(f"\nCorrelações simples com IPV (n = {len(sub7)}):")
for pred in predictors_ipv:
    r, p = stats.pearsonr(sub7[pred], sub7["IPV"])
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "n.s."))
    print(f"  {pred:5s}: r = {r:+.3f}  {sig}")

X7  = StandardScaler().fit_transform(sub7[predictors_ipv].values)
m7  = LinearRegression().fit(X7, sub7["IPV"].values)
r2_7 = m7.score(X7, sub7["IPV"].values)

coef7 = (
    pd.DataFrame({"Indicador": predictors_ipv, "Beta (padronizado)": m7.coef_})
    .sort_values("Beta (padronizado)", ascending=False)
    .reset_index(drop=True)
)
print(f"\nRegressão múltipla — R² = {r2_7:.3f}")
print("Beta padronizado: quanto maior, mais influente no IPV")
print(coef7.to_string(index=False))
print("\n  → IPP e IEG são os maiores preditores positivos do IPV")
print("  → IPS tem efeito negativo leve (aspecto psicossocial elevado ≠ ponto de virada)")


# ══════════════════════════════════════════════════════════════════════════════
# Q8 — MULTIDIMENSIONALIDADE DOS INDICADORES → IMPACTO NO INDE
# ══════════════════════════════════════════════════════════════════════════════
titulo(8, "MULTIDIMENSIONALIDADE DOS INDICADORES — IMPACTO NO INDE")

# ── Regressão completa com coeficientes padronizados ─────────────────────────
features_inde = ["IDA", "IEG", "IPV", "IAN", "IAA", "IPS"]
sub8 = base[["INDE"] + features_inde].dropna()

X8     = StandardScaler().fit_transform(sub8[features_inde].values)
m8     = LinearRegression().fit(X8, sub8["INDE"].values)
r2_8   = m8.score(X8, sub8["INDE"].values)
coefs8 = pd.Series(m8.coef_, index=features_inde).sort_values(ascending=False)

print(f"\n[Regressão completa]  R² = {r2_8:.3f}  (N = {len(sub8)})")
print("Coeficientes padronizados (beta) — maior = mais impacto no INDE:")
for ind, beta in coefs8.items():
    bar = "█" * int(abs(beta) * 20)
    print(f"  {ind}  {beta:+.4f}  {bar}")

# ── Melhor combinação de 2, 3 e 4 variáveis por R² ──────────────────────────
print("\n[Melhor combinação por número de variáveis]")
all_vars_8 = ["IDA", "IEG", "IPS", "IAA", "IPV", "IAN", "IPP"]
for n in [2, 3, 4]:
    best_r2, best_combo = 0.0, None
    for combo in combinations(all_vars_8, n):
        sub_c = base[["INDE"] + list(combo)].dropna()
        if len(sub_c) < 50:
            continue
        Xc   = StandardScaler().fit_transform(sub_c[list(combo)].values)
        r2c  = LinearRegression().fit(Xc, sub_c["INDE"].values).score(Xc, sub_c["INDE"].values)
        if r2c > best_r2:
            best_r2, best_combo = r2c, combo
    print(f"  {n} variáveis: {list(best_combo)}  ->  R² = {best_r2:.3f}")

# ── INDE e IPV médios por perfil IDA × IEG (quadrantes) ─────────────────────
med_ida = base["IDA"].median()
med_ieg = base["IEG"].median()


def classificar_perfil(row):
    alta = row["IDA"] >= med_ida
    alto = row["IEG"] >= med_ieg
    if alta and alto:   return "Alto IDA + Alto IEG"
    if alta and not alto: return "Alto IDA + Baixo IEG"
    if not alta and alto: return "Baixo IDA + Alto IEG"
    return "Baixo IDA + Baixo IEG"


base["Perfil IDA×IEG"] = base.apply(classificar_perfil, axis=1)
print("\n[INDE e IPV médios por perfil IDA × IEG]")
print(
    base.groupby("Perfil IDA×IEG")[["INDE", "IPV"]].mean().round(3)
    .sort_values("INDE", ascending=False).to_string()
)

# ── Com IPP incluído (disponível em 2023–2024) ───────────────────────────────
sub8_ipp = base[base["Ano"].isin([2023, 2024])][["INDE", "IDA", "IEG", "IPS", "IPP"]].dropna()
X8_ipp   = StandardScaler().fit_transform(sub8_ipp[["IDA", "IEG", "IPS", "IPP"]].values)
m8_ipp   = LinearRegression().fit(X8_ipp, sub8_ipp["INDE"].values)
coefs8_ipp = pd.Series(m8_ipp.coef_, index=["IDA", "IEG", "IPS", "IPP"]).sort_values(ascending=False)

print(
    f"\n[Regressão com IPP — 2023–2024]  "
    f"R² = {m8_ipp.score(X8_ipp, sub8_ipp['INDE'].values):.3f}  (N = {len(sub8_ipp)})"
)
print("Coeficientes padronizados:")
for k, v in coefs8_ipp.items():
    print(f"  {k}: {v:+.4f}")


# ══════════════════════════════════════════════════════════════════════════════
# Q10 — EFETIVIDADE DO PROGRAMA: EVOLUÇÃO POR PEDRA E FASE
# ══════════════════════════════════════════════════════════════════════════════
titulo(10, "EFETIVIDADE DO PROGRAMA — EVOLUÇÃO POR PEDRA E FASE")

# ── INDE médio por Pedra × Ano ───────────────────────────────────────────────
print("\n[INDE médio por Pedra e Ano]")
print(
    base.groupby(["Pedra", "Ano"])["INDE"].mean().unstack().round(3)
    .reindex([p for p in PEDRA_ORDER if p in base["Pedra"].dropna().unique()]).to_string()
)

# ── Distribuição % de alunos por Pedra × Ano ─────────────────────────────────
print("\n[% de alunos por Pedra e Ano]")
total_ano = base.groupby("Ano").size()
dist_pct  = (
    base.groupby(["Ano", "Pedra"]).size().unstack(fill_value=0)
    .div(total_ano, axis=0).mul(100).round(1)
)
print(dist_pct.to_string())

# ── Indicadores médios por Pedra ─────────────────────────────────────────────
print("\n[Indicadores médios por Pedra — todos os anos]")
print(
    base.groupby("Pedra")[["INDE", "IDA", "IEG", "IPV", "IAN", "IPS"]].mean().round(3)
    .reindex([p for p in PEDRA_ORDER if p in base["Pedra"].dropna().unique()]).to_string()
)

# ── INDE médio por Fase e Ano ─────────────────────────────────────────────────
print("\n[INDE médio por Fase e Ano]")
fase_inde = (
    base.groupby(["Fase", "Ano"])["INDE"].mean().unstack().round(3)
)
print(fase_inde.reindex([f for f in FASE_ORDER if f in fase_inde.index]).to_string())

# ── Mobilidade de Pedra entre anos consecutivos ──────────────────────────────
PEDRA_ORD = {"Quartzo": 1, "Ágata": 2, "Ametista": 3, "Topázio": 4}


def calcular_mobilidade(df_antes, col_antes, df_depois, col_depois, label):
    p_antes  = df_antes[["RA", "Pedra"]].rename(columns={"Pedra": col_antes})
    p_depois = df_depois[["RA", "Pedra"]].rename(columns={"Pedra": col_depois})
    mob = p_antes.merge(p_depois, on="RA").dropna()
    mob["ord_antes"]  = mob[col_antes].map(PEDRA_ORD)
    mob["ord_depois"] = mob[col_depois].map(PEDRA_ORD)
    n       = len(mob)
    subiu   = (mob["ord_depois"] > mob["ord_antes"]).sum()
    manteve = (mob["ord_depois"] == mob["ord_antes"]).sum()
    caiu    = (mob["ord_depois"] < mob["ord_antes"]).sum()
    print(f"\n  {label}  (N = {n})")
    print(f"    Subiu:    {subiu:3d}  ({subiu / n * 100:.1f}%)")
    print(f"    Manteve:  {manteve:3d}  ({manteve / n * 100:.1f}%)")
    print(f"    Caiu:     {caiu:3d}  ({caiu / n * 100:.1f}%)")


print("\n[Mobilidade de Pedra entre anos consecutivos]")
calcular_mobilidade(df22c, "Pedra_22", df23c, "Pedra_23", "2022 → 2023")
calcular_mobilidade(df23c, "Pedra_23", df24c, "Pedra_24", "2023 → 2024")

# ── Evolução longitudinal: alunos presentes nos 3 anos ───────────────────────
ids3 = set(df22c["RA"]) & set(df23c["RA"]) & set(df24c["RA"])
long = pd.concat([
    df22c[df22c["RA"].isin(ids3)][["RA", "INDE", "IDA", "IEG", "IPV"]].assign(Ano=2022),
    df23c[df23c["RA"].isin(ids3)][["RA", "INDE", "IDA", "IEG", "IPV"]].assign(Ano=2023),
    df24c[df24c["RA"].isin(ids3)][["RA", "INDE", "IDA", "IEG", "IPV"]].assign(Ano=2024),
], ignore_index=True)

print(f"\n[Evolução longitudinal — {len(ids3)} alunos presentes nos 3 anos]")
print(long.groupby("Ano")[["INDE", "IDA", "IEG", "IPV"]].mean().round(3).to_string())


# ══════════════════════════════════════════════════════════════════════════════
# RESUMO EXECUTIVO
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{SEP}")
print("  RESUMO EXECUTIVO")
print(SEP)
print("""
Q1  IAN  : Defasagem cai de -0,94 (2022) para -0,41 (2024). Alunos moderadamente
            defasados caem de 67% para 46%. Melhora consistente e relevante.

Q2  IDA  : IDA cresce de 2022 para 2023 (+9%) mas recua em 2024 (-5%). Fases
            intermediárias (3–5) concentram o pior desempenho. ALFA e FASE 8
            têm os melhores resultados.

Q3  IEG  : Correlação forte com IDA (r=0,54) e IPV (r=0,56). Alunos com
            engajamento alto têm IDA 3,4 pontos acima dos de baixo engajamento.

Q4  IAA  : Autoavaliação descolada do desempenho real (r=0,12). 61,5% dos alunos
            superestimam sua performance em mais de 1 ponto. Sinal de baixa
            metacognição.

Q5  IPS  : Correlação linear fraca com IDA/IEG. Porém, IPS baixo coincide
            com IEG e IPV menores — efeito indireto e não linear.

Q6  IPP  : IPP confirma IDA (r=0,37) com mais força do que confirma IAN
            (r=0,12). Avaliação psicopedagógica capta melhor desempenho do
            que defasagem de nível.

Q7  IPV  : IPP (β=0,47) e IEG (β=0,28) são os maiores preditores positivos
            do ponto de virada. Modelo explica 53% da variância do IPV (R²=0,53).

Q8  INDE : IDA é o maior preditor isolado do INDE (β mais alto). A melhor
            combinação de 3 variáveis é IDA + IEG + IPV, que explica ~90%
            da variância do INDE.

Q10 PROG : Topázio cresce de 15% (2022) para 28% (2024) dos alunos. Nos
            468 alunos longitudinais, INDE sobe de 7,14 para 7,60 (+6,4%).
            Programa demonstra efetividade consistente ao longo dos 3 anos.
""")



  Q1 — ADEQUAÇÃO DO NÍVEL (IAN) E DEFASAGEM ESCOLAR

IAN médio por ano:
Ano
2022    6.424
2023    7.244
2024    7.684

Distribuição de risco IAN por ano (%):
Nível IAN  Severo (<4)  Moderado (4–6)  Adequado (≥8)
Ano                                                  
2022               3.3            66.6           30.1
2023               1.4            53.1           45.6
2024               0.3            45.9           53.8

Defasagem média por ano:
Ano
2022   -0.943
2023   -0.655
2024   -0.409

% alunos sem defasagem (Defasagem = 0) por ano:
Ano
2022    28.7
2023    41.4
2024    42.0

IAN médio por fase (todos os anos):


/tmp/ipykernel_318/2814123329.py:153: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sem_def = base.groupby("Ano").apply(


Fase
ALFA       6.872
FASE 1     6.118
FASE 2     7.171
FASE 3     7.872
FASE 4     7.096
FASE 5     6.811
FASE 6     6.645
FASE 7     8.642
FASE 8    10.000

  Q2 — DESEMPENHO ACADÊMICO (IDA) — EVOLUÇÃO POR FASE E ANO

IDA médio por ano:
Ano
2022    6.093
2023    6.663
2024    6.351

IDA médio por fase (todos os anos):
Fase
ALFA      7.303
FASE 1    6.684
FASE 2    6.187
FASE 3    5.393
FASE 4    5.967
FASE 5    6.140
FASE 6    6.920
FASE 7    5.932
FASE 8    8.000

IDA médio por fase × ano:
Ano      2022   2023   2024
Fase                       
ALFA    7.140  7.422  7.320
FASE 1  6.464  6.814  6.791
FASE 2  5.406  6.737  6.250
FASE 3  5.142  5.747  5.348
FASE 4  6.053  6.004  5.879
FASE 5  5.873  5.905  6.453
FASE 6  6.694  6.809  7.230
FASE 7  5.252  7.810  5.810
FASE 8    NaN    NaN  8.000

Notas médias por disciplina e ano:
        Mat    Por    Ing
Ano                      
2022  5.807  6.321  5.881
2023  6.410  6.817  6.200
2024  6.230  6.176  6.596

  Q3 — ENGAJAMENTO (IEG) × 